# Stages 8 and 10 - Train and tune: M1, A1, A2, M2, T

**Runtime → GPU (T4 or better).** Settings in `configs/runs/`. Each run writes `results/<run_id>/`: config, thresholds tuned on validation, predictions per split, validation metrics. Test scores are computed in Stage 9 only.

- **M1** DeBERTa-v3-small, two heads, context cut first (inputs over 512 tokens)
- **A1** M1 without context (H3)
- **A2** M1 on 1k / 2k / 5k / 10k segments (learning curve)
- **M2** SetFit, cross-fitted on the gold set (never predicts a segment it trained on)
- **T** Claude teacher on the gold set: **costs money**, dry run first

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[ml,label]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch Runtime to GPU")

In [ ]:
from krs.models.runner import run_experiment

m1 = run_experiment("M1")
{k: m1[k] for k in ("macro_f1_knowledge_types", "macro_f1_domains", "macro_f1_mean", "best_epoch")}

In [ ]:
import pandas as pd
pd.DataFrame(m1["history"])   # validation macro-F1 per epoch

In [ ]:
a1 = run_experiment("A1")   # without context (H3)
{"with context (M1)": m1["macro_f1_mean"], "without context (A1)": a1["macro_f1_mean"]}

In [ ]:
a2 = run_experiment("A2")   # learning curve: 4 trainings, the longest step
a2["learning_curve"]

In [ ]:
m2 = run_experiment("M2")   # SetFit, 5-fold cross-fitting on gold
m2

## Teacher T (costs money)

The first cell only estimates. Run the second one after checking the estimate; it needs `ANTHROPIC_API_KEY` in Colab secrets.

In [ ]:
run_experiment("T")   # dry run: estimate only

In [ ]:
# t = run_experiment("T", confirm=True)
# {k: t[k] for k in ("labelled", "failed", "cost_usd", "cost_usd_per_1k", "latency_s_median")}